In [1]:
import time
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import norm

from model import TransferModel
from config import SlimConfig
from data_processing import DataProcessor
from real_score import load_line, set_line_context, score_line   # your copy with the settings loop

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- settings -------------------------------------------------------------
K, LAM, ALPHA = 20, 100, 1.5  # model config you'd submit (see validate_loco.ipynb)
CONTEXT = "A"                  # which context's controls to test against
N_TARGETS = 30                 # sampled challenge targets (more = slower, steadier numbers)
SEED = 2
FDR = 0.05                     # BH threshold for "significant"

In [2]:
DATASETS = {
    "K562": "../../data/replogle_nadig/K562_gwps_raw_bulk_01.h5ad",
    "RPE1": "../../data/replogle_nadig/rpe1_raw_bulk_01.h5ad",
    "HepG2": "../../data/replogle_nadig/nadig_hepg2_pseudobulk.h5ad",
    "Jurkat": "../../data/replogle_nadig/nadig_jurkat_pseudobulk.h5ad",
}

In [3]:
gene_names = pd.read_csv("../../data/controls/gene_names.csv")["gene_name"].tolist()
dp = DataProcessor(gene_names)
kd = 0.9

In [4]:
train_names = list(DATASETS)                                   # all four training lines

model = TransferModel(gene_names, '../../data/embeddings/gene_string_embeddings.v0.3.h5', SlimConfig())
model.config.K, model.config.lambda_reg = K, LAM

measured_genes = {name: dp.load_dataset(name).var_names for name in train_names}
model.fit(*dp.build_Y(train_names, genes="union"), measured_genes)

  Fitted on 15627 perts, STRING coverage: 15118/15627
  direct transfer from ('K562',): 9675 targets have a measurement


In [5]:
xaira_data = load_line('HCT116', gene_names)
vcc2025_data = load_line('H1', gene_names, target_umi = 20000)

lines_dict = {
    'HCT116': xaira_data,
    'H1': vcc2025_data
}

HCT116: 18400 controls, 198 targets (cells per target median 184, min 100), 100 dropped (7s)
H1: 18400 controls, 97 targets (cells per target median 400, min 139), 3 dropped, control UMI/cell median 54616 -> 20009 (p=0.366) (45s)


In [6]:
GRID = {}

for N in (25, 50, 100):
    for top_gain in (2.5, 4):
        for base_gain in (0.5, 1):
            GRID[f'{N}_{top_gain}_{base_gain}'] = {
                "gain": base_gain,
                "n_call": N,
                "gain_top": top_gain
            }
GRID[f'{N}_2.5_2.5'] = {
    "gain": 2.5,
    "n_call": N,
    "gain_top": 2.5
}

GRID

{'25_2.5_0.5': {'gain': 0.5, 'n_call': 25, 'gain_top': 2.5},
 '25_2.5_1': {'gain': 1, 'n_call': 25, 'gain_top': 2.5},
 '25_4_0.5': {'gain': 0.5, 'n_call': 25, 'gain_top': 4},
 '25_4_1': {'gain': 1, 'n_call': 25, 'gain_top': 4},
 '50_2.5_0.5': {'gain': 0.5, 'n_call': 50, 'gain_top': 2.5},
 '50_2.5_1': {'gain': 1, 'n_call': 50, 'gain_top': 2.5},
 '50_4_0.5': {'gain': 0.5, 'n_call': 50, 'gain_top': 4},
 '50_4_1': {'gain': 1, 'n_call': 50, 'gain_top': 4},
 '100_2.5_0.5': {'gain': 0.5, 'n_call': 100, 'gain_top': 2.5},
 '100_2.5_1': {'gain': 1, 'n_call': 100, 'gain_top': 2.5},
 '100_4_0.5': {'gain': 0.5, 'n_call': 100, 'gain_top': 4},
 '100_4_1': {'gain': 1, 'n_call': 100, 'gain_top': 4},
 '100_2.5_2.5': {'gain': 2.5, 'n_call': 100, 'gain_top': 2.5}}

In [7]:
results = {}

for name, line in lines_dict.items():
    set_line_context(model, line, dp, train_names)
    scaled_score, raw_score = score_line(model, line, GRID, kd=kd, alpha=ALPHA, seed=SEED)
    results[name] = [scaled_score, raw_score]

results

HCT116 {'K562': np.float64(0.27), 'RPE1': np.float64(0.3), 'HepG2': np.float64(0.19), 'Jurkat': np.float64(0.24)}
scorer: 10473 DE-tested genes, 18400 controls (6s)
reference + replicate: 121s, median real DE calls 3
25_2.5_0.5: 118s
25_2.5_1: 120s
25_4_0.5: 119s
25_4_1: 120s
50_2.5_0.5: 120s
50_2.5_1: 120s
50_4_0.5: 119s
50_4_1: 119s
100_2.5_0.5: 119s
100_2.5_1: 119s
100_4_0.5: 119s
100_4_1: 119s
100_2.5_2.5: 120s
H1 {'K562': np.float64(0.28), 'RPE1': np.float64(0.23), 'HepG2': np.float64(0.23), 'Jurkat': np.float64(0.25)}
scorer: 10776 DE-tested genes, 18400 controls (13s)
reference + replicate: 135s, median real DE calls 270
25_2.5_0.5: 74s
25_2.5_1: 78s
25_4_0.5: 74s
25_4_1: 70s
50_2.5_0.5: 73s
50_2.5_1: 71s
50_4_0.5: 72s
50_4_1: 74s
100_2.5_0.5: 75s
100_2.5_1: 79s
100_4_0.5: 76s
100_4_1: 77s
100_2.5_2.5: 73s


{'HCT116': [                     pds       mse       fid     reach       jac      nmae  \
  baseline        0.000000  0.000000  0.000000  0.000000  0.000000 -0.000000   
  replicate       1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
  25_2.5_0.5      0.238304  0.340089  0.175780  0.029747  0.021643  0.038387   
  25_2.5_1        0.317395  0.474565  0.155514  0.150698  0.016122  0.088034   
  25_4_0.5        0.280563  1.000000  0.188095  0.038335  0.024818  0.016964   
  25_4_1          0.282502  1.000000  0.160066  0.110489  0.016575  0.061716   
  50_2.5_0.5      0.146549  0.404319  0.112758  0.061262  0.017862  0.057833   
  50_2.5_1        0.079349  0.589289  0.119711  0.153214  0.015679  0.107190   
  50_4_0.5        0.195270  1.000000  0.140947  0.192131  0.018595  0.033269   
  50_4_1          0.093047  1.000000  0.094189  0.236952  0.015932  0.083588   
  100_2.5_0.5     0.162962  0.596346  0.108409  0.113991  0.014635  0.093719   
  100_2.5_1       0.091884  0.

In [8]:
results['H1'][1]

,pds,mse,fid,reach,jac,nmae,n_sig
baseline,0.500000,0.958439,0.550502,0.094440,0.081037,0.948274,10233.0
replicate,0.955649,0.066307,0.859185,0.953375,0.424825,0.351064,84.0
25_2.5_0.5,0.607603,1.063598,0.221883,0.070787,0.006397,0.993608,27.0
25_2.5_1,0.658505,1.088921,0.276533,0.151784,0.017753,0.980881,86.0
25_4_0.5,0.617912,1.150063,0.223202,0.093566,0.006975,0.996849,29.0
25_4_1,0.634558,1.175008,0.278922,0.094743,0.018320,0.983232,85.0
50_2.5_0.5,0.626503,1.094691,0.244465,0.121911,0.008480,0.986509,48.0
50_2.5_1,0.631551,1.120570,0.285434,0.102261,0.019750,0.988104,101.0
50_4_0.5,0.613187,1.227600,0.245852,0.101316,0.009148,0.995399,51.0
50_4_1,0.659579,1.252482,0.286658,0.120596,0.018781,0.988979,104.0


In [9]:
merged = pd.concat(
    {(line, kind): df
     for line, (scaled, raw) in results.items()
     for kind, df in (("scaled", scaled), ("raw", raw))},
    names=["line", "kind", "setting"],
)
merged.reset_index(inplace = True)

In [10]:
merged.sort_values('overall', ascending = False)

,line,kind,setting,pds,mse,fid,reach,jac,nmae,overall,n_sig
32,H1,scaled,replicate,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,84.0
1,HCT116,scaled,replicate,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.0
13,HCT116,scaled,100_4_1,0.306668,1.000000,0.116366,0.178793,0.012971,0.059426,0.279037,124.5
5,HCT116,scaled,25_4_1,0.282502,1.000000,0.160066,0.110489,0.016575,0.061716,0.271891,51.0
14,HCT116,scaled,100_2.5_2.5,0.140863,1.000000,0.041730,0.233722,0.007465,0.180317,0.267349,373.5
...,...,...,...,...,...,...,...,...,...,...,...
57,H1,raw,100_2.5_0.5,0.615013,1.128499,0.268253,0.062968,0.013231,0.998185,NaN,91.0
58,H1,raw,100_2.5_1,0.643686,1.147423,0.296446,0.162147,0.020806,0.984857,NaN,127.0
59,H1,raw,100_4_0.5,0.656465,1.333186,0.274674,0.091267,0.014069,1.005549,NaN,99.0
60,H1,raw,100_4_1,0.671607,1.356394,0.299646,0.094928,0.021879,0.999248,NaN,142.0


In [11]:
merged_no_lines = merged.copy().drop(columns = 'line')

In [12]:
merged_no_lines.groupby(['kind', 'setting']).mean().reset_index().sort_values(by = 'overall', ascending = False)

,kind,setting,pds,mse,fid,reach,jac,nmae,overall,n_sig
30,scaled,replicate,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,43.00
17,scaled,100_2.5_2.5,0.200528,0.500000,-0.179961,0.150663,-0.036931,0.065689,0.116665,528.75
19,scaled,100_4_1,0.341644,0.500000,-0.348150,0.089680,-0.079553,-0.012964,0.081776,133.25
23,scaled,25_4_1,0.288906,0.500000,-0.359869,0.055421,-0.082927,0.001589,0.067187,68.00
27,scaled,50_4_1,0.221636,0.500000,-0.380277,0.133702,-0.082578,0.007714,0.066700,87.50
26,scaled,50_4_0.5,0.221840,0.500000,-0.422994,0.100068,-0.095257,-0.022820,0.046806,52.00
18,scaled,100_4_0.5,0.242966,0.500000,-0.395700,0.058098,-0.090459,-0.036428,0.046413,101.50
16,scaled,100_2.5_1,0.203614,0.374356,-0.355569,0.105379,-0.081115,0.017789,0.044076,116.00
21,scaled,25_2.5_1,0.332631,0.237282,-0.366014,0.108730,-0.083978,0.016717,0.040895,67.50
22,scaled,25_4_0.5,0.269671,0.500000,-0.436109,0.018659,-0.095306,-0.032187,0.037455,28.00


In [13]:
merged[merged['kind'] == 'scaled']

,line,kind,setting,pds,mse,fid,reach,jac,nmae,overall,n_sig
0,HCT116,scaled,baseline,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,10297.0
1,HCT116,scaled,replicate,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.0
2,HCT116,scaled,25_2.5_0.5,0.238304,0.340089,0.175780,0.029747,0.021643,0.038387,0.140658,26.0
3,HCT116,scaled,25_2.5_1,0.317395,0.474565,0.155514,0.150698,0.016122,0.088034,0.200388,49.0
4,HCT116,scaled,25_4_0.5,0.280563,1.000000,0.188095,0.038335,0.024818,0.016964,0.258129,27.0
5,HCT116,scaled,25_4_1,0.282502,1.000000,0.160066,0.110489,0.016575,0.061716,0.271891,51.0
6,HCT116,scaled,50_2.5_0.5,0.146549,0.404319,0.112758,0.061262,0.017862,0.057833,0.133431,48.0
7,HCT116,scaled,50_2.5_1,0.079349,0.589289,0.119711,0.153214,0.015679,0.107190,0.177405,69.0
8,HCT116,scaled,50_4_0.5,0.195270,1.000000,0.140947,0.192131,0.018595,0.033269,0.263369,53.0
9,HCT116,scaled,50_4_1,0.093047,1.000000,0.094189,0.236952,0.015932,0.083588,0.253951,71.0
